In [ ]:
# Phase 1 baseline training: 5-fold CV over the lexically-labeled subset of
# train.csv, efficientnet_b0 + mean-pool, NaN-masked BCE. Internet is ON here
# (unlike the submission notebook) so pretrained ImageNet weights can load.
# GIT_SHA is baked in at push time since Kaggle kernels have no git context.
import glob, os, shutil, sys, time

GIT_SHA = 'eea3101'

src_candidates = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)
comp_candidates = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)
SRC = src_candidates[0]
COMP_DIR = comp_candidates[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

from knee.reports import build_lexical_labels
from knee.dataset import KneeStudyDataset
from knee.model import KneeModel
from knee.train import make_folds, train_one_epoch, evaluate, log_experiment, masked_bce_loss
from knee.metrics import per_label_auc, macro_auc
from knee.infer import LABEL_COLUMNS
print('knee package imported successfully from', PKG)

In [ ]:
import pandas as pd

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
train_series_df = pd.read_csv(f'{COMP_DIR}/train_series.csv')

labels_df = build_lexical_labels(train_df[['StudyInstanceUID', 'Report']])
has_any_label = labels_df[LABEL_COLUMNS].notna().any(axis=1)
study_uids = labels_df.loc[has_any_label, 'StudyInstanceUID'].tolist()
labels_df = labels_df[labels_df['StudyInstanceUID'].isin(study_uids)].reset_index(drop=True)

print(f'{len(train_df)} total studies, {len(study_uids)} with at least one lexical label')
print('per-label non-NaN counts:')
print(labels_df[LABEL_COLUMNS].notna().sum())

In [ ]:
N_FOLDS = 5
SEED = 0

fold_assignment = make_folds(study_uids, n_folds=N_FOLDS, seed=SEED)
fold_of = [fold_assignment[uid] for uid in study_uids]
print(f'{N_FOLDS}-fold assignment done, seed={SEED} -- this must stay frozen for every future experiment')

In [ ]:
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm


class CompactCachedDataset(Dataset):
    """Like knee.dataset.CachedDataset, but stores each cached image as a
    single-channel uint8 array instead of KneeStudyDataset's full float32
    3-channel tensor. The 3 channels KneeStudyDataset produces are identical
    copies (grayscale MRI repeated to satisfy an ImageNet-pretrained
    backbone's input shape), so only one needs to be cached; it's re-expanded
    fresh on every read, which is cheap. Caching the full form for all 2151
    studies here would be ~19GB of RAM (16 slices * 3ch * 224 * 224 * 4 bytes
    each) -- that's what silently OOM-killed the kernel on the previous run,
    with no Python traceback, just "Kernel died". This cuts it to ~1.6GB."""

    def __init__(self, base_dataset):
        self.base = base_dataset
        self._cache = {}

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        if idx not in self._cache:
            image, labels, uid = self.base[idx]
            compact = (image[:, 0, :, :] * 255).round().to(torch.uint8)
            self._cache[idx] = (compact, labels, uid)
        compact, labels, uid = self._cache[idx]
        image = compact.float().div(255.0).unsqueeze(1).repeat(1, 3, 1, 1)
        return image, labels, uid


base_dataset = KneeStudyDataset(
    study_uids=study_uids,
    dcm_root=f'{COMP_DIR}/train_series',
    series_df=train_series_df,
    labels_df=labels_df,
    n_slices=16,
    size=224,
    max_series=1,
)
cached_dataset = CompactCachedDataset(base_dataset)

# One-time decode pass so every fold/epoch after this reads from memory
# instead of re-decoding DICOMs -- decode dominates runtime far more than
# the model forward pass, so this is the single biggest speedup available.
decode_failures = 0
t0 = time.time()
for i in tqdm(range(len(cached_dataset)), desc='Decoding + caching studies'):
    try:
        cached_dataset[i]
    except Exception as e:
        decode_failures += 1
print(f'decode pass: {time.time() - t0:.1f}s, {decode_failures} failures out of {len(cached_dataset)}')

In [ ]:
import numpy as np
from torch.utils.data import DataLoader, Subset

# Kaggle's kernel-metadata has no field to request a specific GPU model --
# whichever accelerator gets assigned (P100, T4, ...) is out of our control,
# and this PyTorch build has dropped support for older architectures (Pascal
# / P100, compute capability sm_60). Rather than crash deep inside the first
# batch norm kernel launch, actually try a real CUDA op up front and fall
# back to CPU if it fails -- allocation alone (torch.zeros(..., device='cuda'))
# can succeed even when kernel execution can't, so this forces a real launch.
device = 'cpu'
if torch.cuda.is_available():
    try:
        _probe = torch.zeros(1, device='cuda') + 1
        device = 'cuda'
    except Exception as e:
        print(f'GPU present ({torch.cuda.get_device_name(0)}) but incompatible with this PyTorch build: {e}')
        print('falling back to CPU training')
print('device:', device)

# Reduced from 3 to 1 epoch/fold for this run since a CPU fallback would be
# meaningfully slower than the GPU run this was tuned for -- bounds the
# worst-case wall time if we land on CPU again. Bump back up once a GPU run
# actually goes through cleanly.
EPOCHS_PER_FOLD = 1
BATCH_SIZE = 8
CONFIG_HASH = f'phase1_baseline_efficientnet_b0_n{len(study_uids)}_e{EPOCHS_PER_FOLD}_{device}'

EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
_REAL_HEADER = (
    'date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,'
    'medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,'
    'pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,'
    'macro_auc,paired_delta,train_minutes,inference_seconds,promoted'
)
with open(EXPERIMENTS_CSV, 'w') as f:
    f.write(_REAL_HEADER + '\n')

oof_true = np.full((len(study_uids), len(LABEL_COLUMNS)), np.nan)
oof_pred = np.full((len(study_uids), len(LABEL_COLUMNS)), np.nan)

for fold in tqdm(range(N_FOLDS), desc='Folds'):
    train_idx = [i for i, f in enumerate(fold_of) if f != fold]
    val_idx = [i for i, f in enumerate(fold_of) if f == fold]

    train_loader = DataLoader(Subset(cached_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(Subset(cached_dataset, val_idx), batch_size=BATCH_SIZE)

    model = KneeModel(backbone_name='efficientnet_b0', num_labels=len(LABEL_COLUMNS), pretrained=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    fold_train_start = time.time()
    for epoch in range(EPOCHS_PER_FOLD):
        loss = train_one_epoch(
            model,
            tqdm(train_loader, desc=f'Fold {fold} epoch {epoch}', leave=False),
            optimizer,
            device=device,
        )
        print(f'fold {fold} epoch {epoch}: loss={loss:.4f}')
    fold_train_minutes = (time.time() - fold_train_start) / 60

    eval_start = time.time()
    y_true, y_pred = evaluate(model, tqdm(val_loader, desc=f'Fold {fold} eval', leave=False), device=device)
    fold_eval_seconds = time.time() - eval_start

    oof_true[val_idx] = y_true
    oof_pred[val_idx] = y_pred

    fold_label_auc = dict(zip(LABEL_COLUMNS, per_label_auc(y_true, y_pred)))
    fold_macro = macro_auc(y_true, y_pred)
    print(f'fold {fold} macro AUC: {fold_macro:.4f}')

    log_experiment(
        EXPERIMENTS_CSV,
        git_sha=GIT_SHA,
        config_hash=CONFIG_HASH,
        hypothesis='Phase 1 baseline: efficientnet_b0, single sagittal fluid-sensitive series, lexical labels',
        fold_set=f'primary_v1_fold{fold}',
        seed=SEED,
        per_label_auc=fold_label_auc,
        macro_auc=fold_macro,
        paired_delta=0.0,
        train_minutes=fold_train_minutes,
        inference_seconds=fold_eval_seconds,
        promoted=(fold == 0),
    )

    if fold == 0:
        torch.save(model.state_dict(), '/kaggle/working/knee_phase1_fold0.pt')
        print('saved fold 0 checkpoint for submission')

In [ ]:
overall_macro = macro_auc(oof_true, oof_pred)
overall_per_label = dict(zip(LABEL_COLUMNS, per_label_auc(oof_true, oof_pred)))

print(f'OOF macro AUC across all {N_FOLDS} folds: {overall_macro:.4f}')
print('OOF per-label AUC:')
for label, auc in overall_per_label.items():
    print(f'  {label}: {auc}')

np.save('/kaggle/working/oof_true.npy', oof_true)
np.save('/kaggle/working/oof_pred.npy', oof_pred)
print('experiments.csv, checkpoint, and OOF arrays saved to /kaggle/working -- download these after the run')